# Практика 19 · Від згортки до уваги> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·> Домашнє завдання: [homework.html](homework.html)> ⏱ **Цей зошит нічого не навчає.** Заміряно: **18 с** на вільній машині й **37 с**> на завантаженій — чотири ядра без відеокарти, в один потік. Уся суть теми в тому,> що головні факти про увагу доводяться алгеброю й випадковими вагами, а не прогонами.Заміри, які ми зробимо:1. **Рецептивне поле** — скільки шарів згортки 3×3 треба, щоб дотягнутись через   увесь кадр 28×28.2. **Увага руками** — усі пʼять кроків формули на чотирьох токенах, звірені   з `nn.MultiheadAttention`.3. **Власна реалізація на numpy** проти бібліотечної через `np.allclose`.4. **Перестановка** — головний факт теми: увага еквіваріантна щодо перестановки.5. **Ділення на √d** — що буває без нього при різних розмірностях.6. **Параметри уваги** — ручний рахунок проти `sum(p.numel())`, і скільки   коштує позиційне кодування справжньому ViT.7. **Ціна й карти уваги до навчання.**Кожне число, яке цитує лекція, друкується тут.

In [ ]:
import math
import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Один потік: на дрібних тензорах він і швидший за чотири, і — головне —
# детермінований. Під кількома потоками порядок додавання float інший,
# і числа пливуть від прогону до прогону.
torch.set_num_threads(1)
np.set_printoptions(precision=4, suppress=True)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## Замір 1 · Скільки шарів згортки треба, щоб дотягнутисьРецептивне поле — ділянка входу, здатна вплинути на одне число на виході. Длязгортки 3×3 із кроком 1 воно росте на два пікселі за шар, тобто після `L` шарівдорівнює `(2L+1) × (2L+1)`. Порахуємо, коли воно нарешті накриє кадр 28×28.

In [ ]:
FRAME = 28

print(f"{'шарів':>6} {'рецептивне поле':>17} {'частка кадру':>14}")
layers_needed = None
for layers in range(1, 15):
    side = 2 * layers + 1
    covered = min(side, FRAME)
    share = covered * covered / (FRAME * FRAME)
    # запамʼятовуємо перший шар, після якого поле накрило 27×27
    if layers_needed is None and side >= 27:
        layers_needed = layers
    print(f"{layers:>6} {str(side) + '×' + str(side):>17} {share:>13.1%}")

print()
print("шарів 3×3 до рецептивного поля 27×27 :", layers_needed)
print("ваг у такому стосі (9 на шар)        :", 9 * layers_needed)
print("а увага накриває всі 49 патчів уже на першому шарі")

## Дані: ті самі шість фігурДатасет той самий, що в блоках 2-3: фігури 28×28 шести класів, згенерованіформулами. Нічого не завантажується. Складність беремо робочу для блоку 4 —шум 0.45, зсув центра ±5.Для уваги кадр ріжеться на **патчі 4×4**: 7×7 = 49 патчів, у кожному 16 чисел.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=5, noise=0.45):
    """Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1."""
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо, щоб мережа не завчила одне-єдине положення предмета
    center_y = size / 2 + rng.integers(-jitter, jitter + 1)
    center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    radius = rng.integers(5, 9)

    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def to_patches(image, patch=4):
    """28×28 → (49, 16): кожен рядок — один патч, розпрямлений у вектор."""
    side = image.shape[0] // patch
    return (image.reshape(side, patch, side, patch)
                 .transpose(0, 2, 1, 3)
                 .reshape(side * side, patch * patch))


rng = np.random.default_rng(42)
picture = draw_shape(1, rng)                 # квадрат — наскрізний приклад зошита
patches = to_patches(picture)

print("зображення :", picture.shape)
print("патчів     :", patches.shape[0], "по", patches.shape[1], "чисел")
print("тобто сітка", int(np.sqrt(patches.shape[0])), "×", int(np.sqrt(patches.shape[0])))

Перетворюємо патчі на токени так само, як це робить справжня мережа: лінійнапроєкція в робочу розмірність 32, потім `LayerNorm`. Нормалізація тут некосметика — без неї скалярні добутки виходять дрібними, softmax дає майжерівномірну кашу, і дивитись немає на що.

In [ ]:
DIM, HEADS, TOKENS = 32, 4, 49

torch.manual_seed(0)
patch_projection = nn.Linear(16, DIM)        # 16 чисел патча → 32 числа токена
token_norm = nn.LayerNorm(DIM)

with torch.no_grad():
    tokens = token_norm(patch_projection(torch.tensor(patches))).unsqueeze(0)

print("токени              :", tuple(tokens.shape))
print("середнє по всіх     :", round(tokens.mean().item(), 4))
print("розкид (std)        :", round(tokens.std().item(), 4))

## Замір 2 · Увага руками на чотирьох токенахТой самий приклад, що в лекції: чотири токени по два числа, матриці ваг одиничні(тобто `Q = K = V = X`). Проходимо всі пʼять кроків формули`softmax(QKᵀ / √d) · V` і друкуємо кожне проміжне число.

In [ ]:
X = np.array([[2.0, 0.0],
              [0.0, 2.0],
              [2.0, 2.0],
              [-2.0, 0.0]], dtype=np.float64)

# крок 1: три ролі того самого входу
queries, keys, values = X, X, X

# крок 2: наскільки схожі — скалярний добуток кожного токена з кожним
scores = queries @ keys.T

# крок 3: поправка на розмірність. d = 2, тому ділимо на корінь із двох
scaled = scores / math.sqrt(X.shape[1])

# крок 4: softmax по рядках. Віднімання максимуму нічого не міняє в результаті,
# але рятує від переповнення при великих оцінках
shifted = np.exp(scaled - scaled.max(axis=1, keepdims=True))
attention = shifted / shifted.sum(axis=1, keepdims=True)

# крок 5: зважена сума значень
output = attention @ values

print("крок 2 · QKᵀ\n", scores, "\n")
print("крок 3 · QKᵀ / √2\n", scaled, "\n")
print("крок 4 · матриця уваги\n", attention)
print("сума кожного рядка:", attention.sum(axis=1), "\n")
print("крок 5 · вихід\n", output)

### Чи те саме порахує бібліотека?`nn.MultiheadAttention` із однією головою й одиничними матрицями ваг має датирівно ті самі числа. Поставимо ваги руками й звіримо — це найдешевша перевіркатого, що ми зрозуміли формулу правильно.

In [ ]:
library_small = nn.MultiheadAttention(embed_dim=2, num_heads=1, batch_first=True)

with torch.no_grad():
    # in_proj_weight складається з трьох матриць Q, K, V одна під одною
    library_small.in_proj_weight.copy_(torch.eye(2).repeat(3, 1))
    library_small.in_proj_bias.zero_()
    library_small.out_proj.weight.copy_(torch.eye(2))
    library_small.out_proj.bias.zero_()

    x_small = torch.tensor(X, dtype=torch.float32).unsqueeze(0)
    library_output, library_attention = library_small(
        x_small, x_small, x_small, need_weights=True)

library_output = library_output[0].numpy()
library_attention = library_attention[0].numpy()

assert np.allclose(attention, library_attention, atol=1e-5), "матриці уваги розійшлися!"
assert np.allclose(output, library_output, atol=1e-5), "виходи розійшлися!"

print("наша матриця уваги      :\n", attention.round(3))
print("бібліотечна             :\n", library_attention.round(3))
print()
print("найбільша розбіжність   :", f"{np.abs(output - library_output).max():.2e}")
print("✅ ручний рахунок збігається з nn.MultiheadAttention")

## Замір 3 · Власна увага на numpy проти бібліотечноїТепер серйозніше: справжні випадкові матриці ваг, 49 токенів, чотири голови.Напишемо багатоголову увагу з нуля на numpy, візьмемо ваги з бібліотечного шаруй перевіримо, що результати збігаються. Усередині бібліотеки немає магії — тамрівно ці десять рядків.

In [ ]:
def attention_numpy(x, w_query, w_key, w_value, w_out,
                    b_query, b_key, b_value, b_out, heads):
    """Багатоголова самоувага. x має форму (токенів, розмірність)."""
    n_tokens, dim = x.shape
    head_dim = dim // heads

    # три описи кожного токена: що я шукаю, що пропоную, що віддам
    q = x @ w_query.T + b_query
    k = x @ w_key.T + b_key
    v = x @ w_value.T + b_value

    result = np.zeros((n_tokens, dim), dtype=np.float64)
    for head in range(heads):
        part = slice(head * head_dim, (head + 1) * head_dim)
        # оцінки збігу всередині своєї голови, з поправкою на її розмірність
        s = (q[:, part] @ k[:, part].T) / math.sqrt(head_dim)
        e = np.exp(s - s.max(axis=1, keepdims=True))
        weights = e / e.sum(axis=1, keepdims=True)
        result[:, part] = weights @ v[:, part]

    # склеєні голови проганяємо через одну вихідну матрицю
    return result @ w_out.T + b_out


torch.manual_seed(3)
library = nn.MultiheadAttention(DIM, HEADS, batch_first=True)

packed_weight = library.in_proj_weight.detach().numpy().astype(np.float64)
packed_bias = library.in_proj_bias.detach().numpy().astype(np.float64)
out_weight = library.out_proj.weight.detach().numpy().astype(np.float64)
out_bias = library.out_proj.bias.detach().numpy().astype(np.float64)

probe = torch.randn(1, TOKENS, DIM)
with torch.no_grad():
    reference = library(probe, probe, probe, need_weights=False)[0][0].numpy()

ours = attention_numpy(
    probe[0].numpy().astype(np.float64),
    packed_weight[:DIM], packed_weight[DIM:2 * DIM], packed_weight[2 * DIM:], out_weight,
    packed_bias[:DIM], packed_bias[DIM:2 * DIM], packed_bias[2 * DIM:], out_bias,
    HEADS)

assert np.allclose(ours, reference, atol=1e-5), "розрахунок розійшовся!"
print("найбільша розбіжність:", f"{np.abs(ours - reference).max():.2e}")
print("✅ збігається")

## Замір 4 · Головний факт теми: увага не бачить порядкуПереставмо токени на вході. Якщо увага **еквіваріантна щодо перестановки**, товихід від перемішаного входу має точно дорівнювати перемішаному виходу відзвичайного входу.Порівняємо три операції на одному й тому самому вході:* увагу без позиційного кодування,* увагу з позиційним кодуванням,* згортку по послідовності токенів (`Conv1d` із ядром 3).

In [ ]:
def sinusoid_positions(n_tokens, dim):
    """Класичне синусоїдальне кодування: жодного параметра, чиста формула."""
    position = torch.arange(n_tokens).unsqueeze(1).float()
    step = torch.arange(0, dim, 2).float()
    angle = position / torch.pow(10000.0, step / dim)
    table = torch.zeros(n_tokens, dim)
    table[:, 0::2] = torch.sin(angle)
    table[:, 1::2] = torch.cos(angle)
    return table.unsqueeze(0)


position_code = sinusoid_positions(TOKENS, DIM)

torch.manual_seed(0)
attention_layer = nn.MultiheadAttention(DIM, HEADS, batch_first=True)
conv_layer = nn.Conv1d(DIM, DIM, kernel_size=3, padding=1)


def run_attention(t):
    with torch.no_grad():
        return attention_layer(t, t, t, need_weights=False)[0]


def run_conv(t):
    # Conv1d чекає (батч, канали, довжина), а токени лежать (батч, довжина, канали)
    with torch.no_grad():
        return conv_layer(t.transpose(1, 2)).transpose(1, 2)


generator = torch.Generator().manual_seed(7)
shuffle = torch.randperm(TOKENS, generator=generator)

plain_normal = run_attention(tokens)
plain_shuffled = run_attention(tokens[:, shuffle, :])
gap_plain = (plain_shuffled - plain_normal[:, shuffle, :]).abs().max().item()

coded_normal = run_attention(tokens + position_code)
coded_shuffled = run_attention(tokens[:, shuffle, :] + position_code)
gap_coded = (coded_shuffled - coded_normal[:, shuffle, :]).abs().max().item()

conv_normal = run_conv(tokens)
conv_shuffled = run_conv(tokens[:, shuffle, :])
gap_conv = (conv_shuffled - conv_normal[:, shuffle, :]).abs().max().item()

print("перестановка (перші 10 позицій):", shuffle[:10].tolist(), "...")
print("масштаб самого виходу          :", round(plain_normal.abs().max().item(), 4))
print()
print(f"увага БЕЗ позиційного кодування: {gap_plain:.3e}   ← нуль із точністю float32")
print(f"увага З позиційним кодуванням  : {gap_coded:.4f}")
print(f"згортка Conv1d(3)              : {gap_conv:.4f}")

### Що з цього виходить для класифікаціїАбстракція стає відчутною на двох картинках. Візьмемо маленьку фігуру у**верхньому лівому** куті та її ж у **нижньому правому**. Друга отримана з першоїчистою перестановкою патчів: жоден піксель не змінився, змінилися лише місця.Проженемо обидві через увагу, усереднимо 49 виходів і подамо в лінійнийкласифікатор. Без позиційного кодування він **не зможе їх розрізнити** — і цене питання навчання чи кількості даних.

In [ ]:
torch.manual_seed(0)
classifier = nn.Linear(DIM, len(SHAPE_NAMES))

blank = np.zeros((28, 28), dtype=np.float32)
yy, xx = np.mgrid[0:28, 0:28]
blank[((yy - 6) ** 2 + (xx - 6) ** 2) < 20] = 1.0     # кружечок угорі ліворуч

top_left = to_patches(blank)
# поворот сітки патчів на 180°: та сама фігура опиняється внизу праворуч
rotate = np.arange(49).reshape(7, 7)[::-1, ::-1].reshape(-1)
bottom_right = top_left[rotate]


def logits_of(patch_rows, with_position):
    with torch.no_grad():
        x = token_norm(patch_projection(torch.tensor(patch_rows))).unsqueeze(0)
        if with_position:
            x = x + position_code
        return classifier(run_attention(x).mean(dim=1))[0].numpy()


for with_position in (False, True):
    a = logits_of(top_left, with_position)
    b = logits_of(bottom_right, with_position)
    label = "З ПОЗИЦІЙНИМ " if with_position else "БЕЗ ПОЗИЦІЙНОГО"
    print(f"{label}: найбільша різниця логітів {np.abs(a - b).max():.3e}")

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, rows, title in zip(axes, [top_left, bottom_right],
                           ["фігура вгорі ліворуч", "ті самі патчі, переставлені"]):
    ax.imshow(rows.reshape(7, 7, 4, 4).transpose(0, 2, 1, 3).reshape(28, 28), cmap="gray_r")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Замір 5 · Навіщо ділити на √dСкалярний добуток двох випадкових векторів довжини `d` має дисперсію рівно `d`.Отже, з ростом розмірності оцінки збігу роздуваються, softmax насичуєтьсяй починає віддавати всю вагу одному елементу.Поміряємо чотири величини для розмірностей 4, 16, 64 і 256:* **дисперсію** самих оцінок;* **найбільшу вагу** в рядку після softmax;* **ентропію** рядка — наскільки розподіл розмазаний (максимум `ln 49 ≈ 3.892`);* **суму `p·(1−p)`** — наскільки softmax узагалі здатний пропустити градієнт.

In [ ]:
def softmax_row_stats(logits):
    """Максимальна вага, ентропія рядка й здатність пропускати градієнт."""
    probabilities = torch.softmax(logits, dim=-1)
    top = probabilities.max(dim=-1).values.mean().item()
    entropy = (-(probabilities * torch.log(probabilities + 1e-12)).sum(-1)).mean().item()
    # p(1-p) — діагональ якобіана softmax: коли вона коло нуля, градієнт не йде
    slope = (probabilities * (1 - probabilities)).sum(-1).mean().item()
    return top, entropy, slope


torch.manual_seed(0)
SAMPLES = 200

print(f"{'d':>5} {'дисперс.':>9} {'макс без':>9} {'макс з':>8} "
      f"{'ентроп. без':>12} {'ентроп. з':>10} {'нахил без':>10} {'нахил з':>8} "
      f"{'у скільки гірше':>8}")
for dim in (4, 16, 64, 256):
    q = torch.randn(SAMPLES, TOKENS, dim)
    k = torch.randn(SAMPLES, TOKENS, dim)
    raw = q @ k.transpose(1, 2)
    scaled_logits = raw / math.sqrt(dim)

    top_raw, ent_raw, slope_raw = softmax_row_stats(raw)
    top_sc, ent_sc, slope_sc = softmax_row_stats(scaled_logits)

    print(f"{dim:>5} {raw.var().item():>9.2f} {top_raw:>9.3f} {top_sc:>8.3f} "
          f"{ent_raw:>12.3f} {ent_sc:>10.3f} {slope_raw:>10.3f} {slope_sc:>8.3f} "
          f"{slope_sc / slope_raw:>8.1f}")

print()
print("рядків у вибірці       :", SAMPLES * TOKENS)
print("рівномірна вага (1/49) :", round(1 / TOKENS, 4))
print("максимальна ентропія   :", round(math.log(TOKENS), 3))

Прочитай другу колонку: дисперсія збігається з `d` до другого знака — теоріяпідтверджена. Далі головне: без ділення при `d = 256` один елемент забираєблизько 90 % ваги рядка з 49, а здатність пропускати градієнт падає майжевсемеро. З діленням усі три величини стоять на місці при будь-якій розмірності.## Замір 6 · Скільки коштує увага в параметрахМатриць чотири: `W_q`, `W_k`, `W_v` і вихідна, кожна `d × d` плюс зсув на `d`чисел. Разом `4·d² + 4·d`. Найцікавіше — що це число **не залежить від кількостіголів**: голови ділять ті самі матриці, а не додають нові.

In [ ]:
print(f"{'d':>5} {'голів':>6} {'розмір голови':>14} {'ручний рахунок':>15} "
      f"{'sum(p.numel())':>15} {'збіг':>6}")
for dim in (32, 64, 768):
    for heads in (1, 2, 4, 8, 12):
        if dim % heads:
            continue
        block = nn.MultiheadAttention(dim, heads, batch_first=True)
        measured = sum(p.numel() for p in block.parameters())
        by_hand = 4 * dim * dim + 4 * dim
        assert by_hand == measured, "формула розійшлася з бібліотекою!"
        print(f"{dim:>5} {heads:>6} {dim // heads:>14} {by_hand:>15,} "
              f"{measured:>15,} {'так':>6}".replace(",", " "))

print()
print("✅ 4·d² + 4·d збігається з sum(p.numel()) у всіх випадках")

### Чи справді голови дивляться в різні місцяПорахуємо, скільки різних цілей (позицій із найбільшою вагою) дають голови длякожного токена, і наскільки корельовані їхні матриці уваги.

In [ ]:
carrier = tokens + position_code

print(f"{'голів':>6} {'розмір голови':>14} {'різних цілей':>13} {'кореляція голів':>16}")
for heads in (1, 2, 4, 8, 16):
    torch.manual_seed(11)
    block = nn.MultiheadAttention(DIM, heads, batch_first=True)
    with torch.no_grad():
        _, per_head = block(carrier, carrier, carrier,
                            need_weights=True, average_attn_weights=False)
    per_head = per_head[0]                      # (голів, токенів, токенів)

    best = per_head.argmax(dim=-1)
    distinct = sum(len(set(best[:, i].tolist())) for i in range(TOKENS)) / TOKENS

    if heads > 1:
        flat = per_head.reshape(heads, -1)
        flat = flat - flat.mean(dim=1, keepdim=True)
        norm = flat.norm(dim=1, keepdim=True)
        matrix = (flat @ flat.T) / (norm @ norm.T)
        # середнє по недіагональних клітинках: наскільки голови схожі між собою
        off_diagonal = ((matrix.sum() - matrix.diag().sum()) / (heads * heads - heads)).item()
        shown = f"{off_diagonal:>16.3f}"
    else:
        shown = f"{'—':>16}"

    print(f"{heads:>6} {DIM // heads:>14} {distinct:>13.2f}{shown}")

## Замір 6б · Скільки коштує геометрія справжньому ViTРаз увага не бачить порядку, положення доводиться дописувати окремою таблицею«номер патча → вектор». Подивимось, скільки вона важить у справжній моделі.Беремо `vit_b_16` із `torchvision` з `weights=None` — нічого не завантажується,модель просто збирається з випадковими вагами, а нам потрібні лише розміри.

In [ ]:
from torchvision.models import vit_b_16

vit = vit_b_16(weights=None)
total_weights = sum(p.numel() for p in vit.parameters())
position_weights = vit.encoder.pos_embedding.numel()

print("ViT-B/16, усього ваг    :", f"{total_weights:,}".replace(",", " "))
print("позиційне кодування     :", f"{position_weights:,}".replace(",", " "),
      "  форма", tuple(vit.encoder.pos_embedding.shape))
print("це", f"{100 * position_weights / total_weights:.3f} %", "усієї моделі")
print()
# 197 позицій = 196 патчів 16×16 на кадрі 224×224 плюс один службовий токен
print("перевірка рахунком: 197 × 768 =", 197 * 768)
assert position_weights == 197 * 768, "розклад позиційного кодування не зійшовся!"
print("✅ уся геометрія моделі — оці 0.175 % ваг")

## Замір 7 · Ціна росте квадратичноМатриця уваги має `n²` клітинок, тож множень у ній `2·n²·d` (оцінки збігу плюсзважена сума значень). Проєкції коштують `4·n·d²` і ростуть **лінійно**.Порахуємо обидва числа для різних розмірів патча — і знайдемо точку, де вониміняються місцями.

In [ ]:
print(f"{'патч':>7} {'токенів':>8} {'пар уваги':>10} {'множень: матриця':>17} "
      f"{'множень: проєкції':>18}")
for patch in (14, 7, 4, 2):
    n = (FRAME // patch) ** 2
    matrix_cost = 2 * n * n * DIM
    projection_cost = 4 * n * DIM * DIM
    print(f"{str(patch) + '×' + str(patch):>7} {n:>8} {n * n:>10} "
          f"{matrix_cost:>17,} {projection_cost:>18,}".replace(",", " "))

print()
print("рівновага 2n²d = 4nd² настає при n = 2d =", 2 * DIM, "токенів")

Тепер те саме годинником. Заміряємо прямий прохід `nn.MultiheadAttention` прирізній кількості токенів. Беремо **мінімум** із семи прогонів, а не середнє:мінімум менше залежить від того, чим ще зайнята машина.

In [ ]:
timed_layer = nn.MultiheadAttention(64, 4, batch_first=True).eval()

print(f"{'токенів':>8} {'пар':>10} {'мс':>9} {'час × від 49':>13} {'пар × від 49':>13}")
baseline = None
for n_tokens in (49, 196, 784, 1568, 3136):
    sample = torch.randn(1, n_tokens, 64)
    repeats = max(3, int(2000 / n_tokens))
    with torch.no_grad():
        for _ in range(5):                      # прогріваємо, щоб не міряти перший виклик
            timed_layer(sample, sample, sample, need_weights=False)
        runs = []
        for _ in range(7):
            started = time.perf_counter()
            for _ in range(repeats):
                timed_layer(sample, sample, sample, need_weights=False)
            runs.append((time.perf_counter() - started) / repeats * 1000)
    fastest = min(runs)
    if baseline is None:
        baseline = (fastest, n_tokens * n_tokens)
    print(f"{n_tokens:>8} {n_tokens * n_tokens:>10} {fastest:>9.3f} "
          f"{fastest / baseline[0]:>13.1f} {n_tokens * n_tokens / baseline[1]:>13.1f}")

Зверни увагу на чесну деталь: між 49 і 196 токенами пар стає в 16 разів більше,а часу — лише разів у півтора. Квадратичність нікуди не поділася, просто на такихрозмірах увесь час зʼїдають накладні витрати виклику. Від 784 токенів починаєтьсячистий квадратичний режим.## Замір 8 · Карти уваги до навчанняОстаннє — щоб не вважати карти уваги магією. Візьмемо шар зі щойно створенимивипадковими вагами й подивимось, куди він дивиться.

In [ ]:
torch.manual_seed(0)
fresh = nn.MultiheadAttention(DIM, HEADS, batch_first=True)
with torch.no_grad():
    _, attention_map = fresh(carrier, carrier, carrier,
                             need_weights=True, average_attn_weights=True)
attention_map = attention_map[0].numpy()

row_entropy = -(attention_map * np.log(attention_map + 1e-12)).sum(axis=1).mean()
uniform_weight = 1 / TOKENS

print("найбільша вага в рядку, у середньому:", round(attention_map.max(axis=1).mean(), 4))
print("рівномірна вага 1/49               :", round(uniform_weight, 4))
print("ентропія рядка                     :", round(row_entropy, 3),
      "з максимальних", round(math.log(TOKENS), 3))
print("це", f"{100 * row_entropy / math.log(TOKENS):.1f} %", "від ідеально рівномірного розподілу")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].imshow(attention_map, cmap="magma")
axes[0].set_title("матриця уваги 49×49 на випадкових вагах", fontsize=9)
axes[0].set_xlabel("у кого бере"); axes[0].set_ylabel("хто бере")
axes[1].bar(range(TOKENS), attention_map[24], width=0.9)
axes[1].axhline(uniform_weight, linestyle="--", linewidth=1)
axes[1].set_title("рядок токена № 24; пунктир — рівномірний рівень", fontsize=9)
axes[1].set_xlabel("номер патча")
plt.tight_layout(); plt.show()

Ентропія коло 99 % від максимальної означає, що до навчання увага розмазанамайже рівномірно. Красиві карти уваги зі статей — це результат навчання, а невластивість самої операції.## Підсумок

In [ ]:
print("Що ми довели без жодного навчання:")
print()
print(f"  1. згортці 3×3 треба {layers_needed} шарів на кадр 28×28, увазі — один")
print(f"  2. ручний рахунок збігається з бібліотекою до {np.abs(output - library_output).max():.1e}")
print(f"  3. власна реалізація на numpy збігається до {np.abs(ours - reference).max():.1e}")
print(f"  4. перестановка: увага {gap_plain:.1e}, з кодуванням {gap_coded:.3f}, "
      f"згортка {gap_conv:.3f}")
print("  5. без ділення на √d softmax при d = 256 віддає ~90 % ваги одному елементу")
print("  6. параметрів 4d² + 4d при будь-якій кількості голів")
print(f"  7. позиційне кодування ViT-B/16 — {position_weights} ваг, "
      f"{100 * position_weights / total_weights:.3f} % моделі")
print()
print("час виконання зошита:", round(time.perf_counter() - notebook_started, 1), "с")

## Завдання### 🟢 Рівень 1Заміни фігуру в наскрізному прикладі (`draw_shape(1, rng)` → інший клас) іповтори замір перестановки. Переконайся, що число «без позиційного кодування»лишається на рівні `1e-07` незалежно від картинки, а число для згортки міняється.### 🟡 Рівень 2Додай до заміру 5 розмірність `d = 1024` і побудуй графік «найбільша вагав рядку від `d`» для обох варіантів — з діленням і без. Признач вісь `d`логарифмічною. Поясни словами, чому крива «з діленням» горизонтальна.### 🔴 Рівень 3Допиши `attention_numpy` так, щоб вона приймала **маску**: булеву матрицю`(токенів, токенів)`, де `True` означає «сюди дивитись не можна». Забороненіклітинки перед softmax отримують `-inf`. Перевір на трикутній масці (кожен токенбачить лише себе й попередників), що результат збігається з`nn.MultiheadAttention(..., attn_mask=...)` через `np.allclose`.